In [1]:
import os
import warnings

#os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import tensorflow as tf

from typing import Any, Dict, List, Optional, Tuple

from tf_agents.agents.ddpg import actor_network
from tf_agents.agents.ddpg import critic_network
from tf_agents.agents.ddpg import ddpg_agent
from tf_agents.drivers import dynamic_step_driver
from tf_agents.environments import py_environment, tf_py_environment
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.specs import array_spec
from tf_agents.trajectories import time_step as ts
from tf_agents.utils import common
from tf_agents.policies import random_tf_policy
from tf_agents.agents.td3 import td3_agent
from tf_agents.agents.sac import sac_agent, tanh_normal_projection_network
from tf_agents.agents.ppo import ppo_clip_agent
from tf_agents.networks import actor_distribution_network, value_network

tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)
if hasattr(tf, "compat"):
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:

class EnergyManagementEnv(py_environment.PyEnvironment):
    """
    Single source of truth for

    1. dataset splitting
    2. BESS sizing
    3. reward standardization
    4. observation normalization
    5. battery dynamics
    6. reward and history accounting

    This removes duplicated accounting logic from the benchmark classes.
    """

    TRAIN_END = 17520
    EVAL_END = 35088
    TEST_END = 52608
    FORECAST_HORIZON = 48

    @staticmethod
    def prepare_buildings(
        path_energy_data: str,
        num_buildings: int,
        ecoPriority: float,
        feed_in_price: float,
    ) -> Tuple[
        Dict[str, Dict[str, pd.DataFrame]],
        Dict[str, Dict[str, Dict[str, float]]],
        Dict[str, Dict[str, float]],
    ]:
        energy_data = pd.read_csv(path_energy_data).fillna(0.0)

        if "Date" in energy_data.columns:
            energy_data = energy_data.drop(columns=["Date"])

        dataset: Dict[str, Dict[str, pd.DataFrame]] = {"train": {}, "eval": {}, "test": {}}
        bess_meta: Dict[str, Dict[str, Dict[str, float]]] = {"train": {}, "eval": {}, "test": {}}
        obs_stats: Dict[str, Dict[str, float]] = {}

        for building_idx in range(1, num_buildings + 1):
            building_key = f"building_{building_idx}"
            load_col = f"load_{building_idx}"
            pv_col = f"pv_{building_idx}"

            building_df = energy_data[[load_col, pv_col, "price", "emissions"]].copy().reset_index(drop=True)

            dataset["train"][building_key] = building_df.iloc[:EnergyManagementEnv.TRAIN_END].reset_index(drop=True)
            dataset["eval"][building_key] = building_df.iloc[EnergyManagementEnv.TRAIN_END:EnergyManagementEnv.EVAL_END].reset_index(drop=True)
            dataset["test"][building_key] = building_df.iloc[EnergyManagementEnv.EVAL_END:EnergyManagementEnv.TEST_END].reset_index(drop=True)

            train_df = dataset["train"][building_key]

            sizing_stats = EnergyManagementEnv.size_battery_from_training_data(train_df)
            reward_stats = EnergyManagementEnv.compute_reward_standardization_stats(
                train_df,
                ecoPriority=ecoPriority,
                feed_in_price=feed_in_price,
            )
            observation_stats = EnergyManagementEnv.compute_observation_normalization_stats(train_df)

            obs_stats[building_key] = observation_stats

            for split in ("train", "eval", "test"):
                bess_meta[split][building_key] = {**sizing_stats, **reward_stats}

        return dataset, bess_meta, obs_stats

    @staticmethod
    def compute_daily_shiftable_energy_kwh(data: pd.DataFrame) -> np.ndarray:
        net_load_kwh = data.iloc[:, 0].to_numpy(dtype=np.float32) - data.iloc[:, 1].to_numpy(dtype=np.float32)
        full_days = len(net_load_kwh) // 48
        if full_days == 0:
            raise ValueError("Need at least one full day of data to size the battery.")
        net_load_kwh = net_load_kwh[: full_days * 48].reshape(full_days, 48)
        daily_surplus_kwh = np.maximum(-net_load_kwh, 0.0).sum(axis=1)
        daily_deficit_kwh = np.maximum(net_load_kwh, 0.0).sum(axis=1)
        return np.minimum(daily_surplus_kwh, daily_deficit_kwh).astype(np.float32)

    @staticmethod
    def size_battery_from_training_data(data: pd.DataFrame) -> Dict[str, float]:
        shiftable_daily_energy_kwh = EnergyManagementEnv.compute_daily_shiftable_energy_kwh(data)
        capacity_kwh = max(float(np.mean(shiftable_daily_energy_kwh)), 1e-6)
        power_battery_kw = capacity_kwh / 3.0
        return {
            "capacity_kwh": capacity_kwh,
            "power_battery_kw": power_battery_kw,
            "storage_duration_hours": capacity_kwh / power_battery_kw,
            "shiftable_daily_energy_mean_kwh": float(np.mean(shiftable_daily_energy_kwh)),
            "shiftable_daily_energy_std_kwh": float(np.std(shiftable_daily_energy_kwh)),
            "shiftable_daily_energy_max_kwh": float(np.max(shiftable_daily_energy_kwh)),
        }

    @staticmethod
    def _compute_robust_stats(
        values: np.ndarray,
        lower_q: float = 0.01,
        upper_q: float = 0.99,
    ) -> Dict[str, float]:
        values = np.asarray(values, dtype=np.float32).reshape(-1)

        clip_low = float(np.quantile(values, lower_q))
        clip_high = float(np.quantile(values, upper_q))
        clipped = np.clip(values, clip_low, clip_high)

        center = float(np.median(clipped))
        scale = float(np.quantile(clipped, 0.75) - np.quantile(clipped, 0.25))

        if scale < 1e-6:
            scale = float(np.std(clipped))
        if scale < 1e-6:
            scale = 1.0

        return {
            "clip_low": clip_low,
            "clip_high": clip_high,
            "center": center,
            "scale": scale,
        }

    @staticmethod
    def compute_observation_normalization_stats(data: pd.DataFrame) -> Dict[str, float]:
        load_kwh = data.iloc[:, 0].to_numpy(dtype=np.float32)
        pv_kwh = data.iloc[:, 1].to_numpy(dtype=np.float32)
        price = data.iloc[:, 2].to_numpy(dtype=np.float32)
        emissions = data.iloc[:, 3].to_numpy(dtype=np.float32)

        net_load_kwh = load_kwh - pv_kwh

        net_load_stats = EnergyManagementEnv._compute_robust_stats(net_load_kwh)
        price_stats = EnergyManagementEnv._compute_robust_stats(price)
        emission_stats = EnergyManagementEnv._compute_robust_stats(emissions)

        return {
            "net_load_clip_low": net_load_stats["clip_low"],
            "net_load_clip_high": net_load_stats["clip_high"],
            "net_load_center": net_load_stats["center"],
            "net_load_scale": net_load_stats["scale"],
            "price_clip_low": price_stats["clip_low"],
            "price_clip_high": price_stats["clip_high"],
            "price_center": price_stats["center"],
            "price_scale": price_stats["scale"],
            "emission_clip_low": emission_stats["clip_low"],
            "emission_clip_high": emission_stats["clip_high"],
            "emission_center": emission_stats["center"],
            "emission_scale": emission_stats["scale"],
        }

    @staticmethod
    def compute_reward_standardization_stats(
        data: pd.DataFrame,
        ecoPriority: float = 0.0,
        feed_in_price: float = 0.076,
    ) -> Dict[str, float]:
        load_kwh = data.iloc[:, 0].to_numpy(dtype=np.float32)
        pv_kwh = data.iloc[:, 1].to_numpy(dtype=np.float32)
        price = data.iloc[:, 2].to_numpy(dtype=np.float32)
        emissions = data.iloc[:, 3].to_numpy(dtype=np.float32)

        net_load_kwh = load_kwh - pv_kwh
        grid_buy_kwh = np.maximum(net_load_kwh, 0.0)
        grid_sell_kwh = np.maximum(-net_load_kwh, 0.0)

        baseline_profit = grid_sell_kwh * feed_in_price - grid_buy_kwh * price
        baseline_emissions = grid_buy_kwh * emissions

        baseline_signal_magnitude = (
            (1.0 - ecoPriority) * np.abs(baseline_profit)
            + ecoPriority * np.abs(baseline_emissions)
        )

        stats = EnergyManagementEnv._compute_robust_stats(baseline_signal_magnitude)

        return {
            "reward_mean": 0.0,
            "reward_std": float(max(stats["scale"], 1e-6)),
        }

    def __init__(
        self,
        data: pd.DataFrame,
        bess_meta: Dict[str, float],
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        forecast_df: Optional[pd.DataFrame] = None,
        obs_stats: Optional[Dict[str, float]] = None,
        track_history: bool = False,
        noise_std: float = 0.0,
        random_seed: int = 42,
        price_feature_mode: str = "local_absolute",
        local_price_lower_q: float = 0.05,
        local_price_upper_q: float = 0.95,
    ):
        super().__init__()

        self._data = data.reset_index(drop=True).copy()
        self._forecast_df = None if forecast_df is None else forecast_df.reset_index(drop=True).copy()

        self._capacity_kwh = float(bess_meta["capacity_kwh"])
        self._power_battery_kw = float(bess_meta["power_battery_kw"])
        self._reward_mean = float(bess_meta["reward_mean"])
        self._reward_std = float(max(bess_meta["reward_std"], 1e-6))

        self._ecoPriority = float(ecoPriority)
        self._feed_in_price = float(feed_in_price)
        self._power_grid_kw = float(power_grid_kw)
        self._grid_import_limit_kwh = self._power_grid_kw * 0.5
        self._init_charge_kwh = float(init_charge_kwh)
        self._penalty_factor = float(penalty_factor)
        self._obs_stats = obs_stats
        self._track_history = bool(track_history)

        self._noise_std = float(noise_std)
        self._rng = np.random.default_rng(random_seed)

        self._price_feature_mode = str(price_feature_mode)
        self._local_price_lower_q = float(local_price_lower_q)
        self._local_price_upper_q = float(local_price_upper_q)

        self._prosumption_cols = [f"prosumption_f{i}" for i in range(1, self.FORECAST_HORIZON + 1)]
        self._price_cols = [f"price_f{i}" for i in range(1, self.FORECAST_HORIZON + 1)]
        self._emission_cols = [f"emission_f{i}" for i in range(1, self.FORECAST_HORIZON + 1)]

        if self._forecast_df is not None:
            required_cols = self._prosumption_cols + self._price_cols + self._emission_cols
            missing_cols = [c for c in required_cols if c not in self._forecast_df.columns]
            if missing_cols:
                raise ValueError(f"forecast_df missing columns: {missing_cols}")
            if len(self._forecast_df) < len(self._data):
                raise ValueError("forecast_df must be at least as long as data.")

        eps = 1e-8
        self._use_price_features = self._ecoPriority < (1.0 - eps)
        self._use_emission_features = self._ecoPriority > eps

        base_dim = 4
        if self._use_price_features:
            base_dim += 1
        if self._use_emission_features:
            base_dim += 1

        forecast_dim = self.FORECAST_HORIZON
        if self._use_price_features:
            forecast_dim += self.FORECAST_HORIZON
        if self._use_emission_features:
            forecast_dim += self.FORECAST_HORIZON

        self._observation_dim = base_dim + forecast_dim

        self._action_spec = array_spec.BoundedArraySpec(
            shape=(1,),
            dtype=np.float32,
            minimum=-1.0,
            maximum=1.0,
            name="action",
        )
        self._observation_spec = array_spec.ArraySpec(
            shape=(self._observation_dim,),
            dtype=np.float32,
            name="observation",
        )

        self._max_timesteps = len(self._data)
        self._current_timestep = 0
        self._episode_ended = False
        self._soe_kwh = float(np.clip(self._init_charge_kwh, 0.0, self._capacity_kwh))
        self._total_profit = 0.0
        self._total_emissions = 0.0
        self._history: List[Dict[str, Any]] = []

    @property
    def soe_kwh(self) -> float:
        return float(self._soe_kwh)

    @property
    def capacity_kwh(self) -> float:
        return float(self._capacity_kwh)

    @property
    def power_battery_kw(self) -> float:
        return float(self._power_battery_kw)

    @property
    def total_profit(self) -> float:
        return float(self._total_profit)

    @property
    def total_emissions(self) -> float:
        return float(self._total_emissions)

    @property
    def history(self) -> List[Dict[str, Any]]:
        return list(self._history)

    def action_spec(self):
        return self._action_spec

    def observation_spec(self):
        return self._observation_spec

    def history_df(self) -> pd.DataFrame:
        return pd.DataFrame(self._history)

    def summary(self,agent_name: str,building_key: str,runtime_seconds: float,training_seconds: float = 0.0,extra_metrics: Optional[Dict[str, Any]] = None,) -> Dict[str, Any]:
        decision_times = []
        for row in self._history:
            dt = row.get("decision_time_seconds", np.nan)
            if dt is not None and np.isfinite(dt):
                decision_times.append(float(dt))

        inference_total_seconds = float(np.sum(decision_times)) if len(decision_times) > 0 else 0.0
        inference_mean_seconds = float(np.mean(decision_times)) if len(decision_times) > 0 else 0.0
        inference_median_seconds = float(np.median(decision_times)) if len(decision_times) > 0 else 0.0
        inference_max_seconds = float(np.max(decision_times)) if len(decision_times) > 0 else 0.0

        summary_dict = {
            "Agent": agent_name,
            "Building": building_key,
            "Total Profit": float(self._total_profit),
            "Total Cost": float(-self._total_profit),
            "Total Emissions": float(self._total_emissions),
            "Total Raw Reward": float(sum(row["raw_reward"] for row in self._history)),
            "RuntimeSeconds": float(runtime_seconds),
            "TrainingSeconds": float(training_seconds),
            "InferenceTotalSeconds": inference_total_seconds,
            "InferenceMeanSeconds": inference_mean_seconds,
            "InferenceMedianSeconds": inference_median_seconds,
            "InferenceMaxSeconds": inference_max_seconds,
            "NumInferenceDecisions": int(len(decision_times)),
        }

        if extra_metrics is not None:
            summary_dict.update(extra_metrics)

        return summary_dict
    
    def _reset(self):
        self._current_timestep = 0
        self._episode_ended = False
        self._soe_kwh = float(np.clip(self._init_charge_kwh, 0.0, self._capacity_kwh))
        self._total_profit = 0.0
        self._total_emissions = 0.0
        self._history = []
        return ts.restart(self._build_observation(0))

    def _step(self, action):
        if self._episode_ended:
            return self.reset()

        t = self._current_timestep

        action_norm = float(np.clip(np.asarray(action).reshape(-1)[0], -1.0, 1.0))
        requested_battery_power_kw = action_norm * self._power_battery_kw
        requested_delta_soe_kwh = requested_battery_power_kw * 0.5

        soe_old_kwh = float(self._soe_kwh)
        soe_new_kwh = float(np.clip(soe_old_kwh + requested_delta_soe_kwh, 0.0, self._capacity_kwh))
        actual_delta_soe_kwh = soe_new_kwh - soe_old_kwh
        actual_battery_power_kw = actual_delta_soe_kwh / 0.5
        self._soe_kwh = soe_new_kwh

        penalty_soe = abs(requested_delta_soe_kwh - actual_delta_soe_kwh) * self._penalty_factor
        penalty_aging = 0.0

        load_kwh = float(self._data.iloc[t, 0])
        pv_kwh = float(self._data.iloc[t, 1])
        price = float(self._data.iloc[t, 2])
        emissions = float(self._data.iloc[t, 3])
        net_load_kwh = load_kwh - pv_kwh

        grid_energy_kwh = net_load_kwh + actual_delta_soe_kwh
        grid_buy_kwh = max(grid_energy_kwh, 0.0)
        grid_sell_kwh = max(-grid_energy_kwh, 0.0)

        grid_violation_kwh = max(grid_buy_kwh - self._grid_import_limit_kwh, 0.0)
        penalty_grid = grid_violation_kwh * self._penalty_factor

        step_profit = grid_sell_kwh * self._feed_in_price - grid_buy_kwh * price
        step_emissions = grid_buy_kwh * emissions

        self._total_profit += step_profit
        self._total_emissions += step_emissions

        baseline_grid_buy_kwh = max(net_load_kwh, 0.0)
        baseline_grid_sell_kwh = max(-net_load_kwh, 0.0)
        baseline_step_profit = baseline_grid_sell_kwh * self._feed_in_price - baseline_grid_buy_kwh * price
        baseline_step_emissions = baseline_grid_buy_kwh * emissions

        reward_profit_component = step_profit - baseline_step_profit
        reward_emission_component = baseline_step_emissions - step_emissions

        raw_reward = (
            (1.0 - self._ecoPriority) * reward_profit_component
            + self._ecoPriority * reward_emission_component
        )
        reward = (raw_reward - self._reward_mean) / self._reward_std

        if self._track_history:
            self._history.append(
                {
                    "timestep": t,
                    "action_normalized": action_norm,
                    "requested_battery_power_kw": requested_battery_power_kw,
                    "actual_battery_power_kw": actual_battery_power_kw,
                    "requested_delta_soe_kwh": requested_delta_soe_kwh,
                    "actual_delta_soe_kwh": actual_delta_soe_kwh,
                    "soe_kwh": self._soe_kwh,
                    "soc": self._soe_kwh / self._capacity_kwh if self._capacity_kwh > 0.0 else 0.0,
                    "capacity_kwh": self._capacity_kwh,
                    "power_battery_kw": self._power_battery_kw,
                    "load_kwh": load_kwh,
                    "pv_kwh": pv_kwh,
                    "net_load_kwh": net_load_kwh,
                    "grid_energy_kwh": grid_energy_kwh,
                    "grid_buy_kwh": grid_buy_kwh,
                    "grid_sell_kwh": grid_sell_kwh,
                    "price": price,
                    "grid_emissions": emissions,
                    "baseline_step_profit": baseline_step_profit,
                    "baseline_step_emissions": baseline_step_emissions,
                    "reward_profit_component": reward_profit_component,
                    "reward_emission_component": reward_emission_component,
                    "step_profit": step_profit,
                    "total_profit": self._total_profit,
                    "step_emissions": step_emissions,
                    "total_emissions": self._total_emissions,
                    "penalty_soe": penalty_soe,
                    "penalty_aging": penalty_aging,
                    "penalty_grid": penalty_grid,
                    "raw_reward": raw_reward,
                    "reward": reward,
                }
            )

        next_t = t + 1

        if next_t >= self._max_timesteps:
            self._episode_ended = True
            return ts.termination(
                observation=self._build_observation(self._max_timesteps - 1),
                reward=np.float32(reward),
            )

        self._current_timestep = next_t
        return ts.transition(
            observation=self._build_observation(self._current_timestep),
            reward=np.float32(reward),
        )

    def _sample_noise(self, size: int) -> np.ndarray:
        if self._noise_std <= 0.0:
            return np.zeros(size, dtype=np.float32)
        return self._rng.normal(0.0, self._noise_std, size=size).astype(np.float32)

    def _robust_normalize(self, values: np.ndarray, feature_name: str) -> np.ndarray:
        values = np.asarray(values, dtype=np.float32)
        if self._obs_stats is None:
            return values
        clip_low = self._obs_stats[f"{feature_name}_clip_low"]
        clip_high = self._obs_stats[f"{feature_name}_clip_high"]
        center = self._obs_stats[f"{feature_name}_center"]
        scale = self._obs_stats[f"{feature_name}_scale"]
        return ((np.clip(values, clip_low, clip_high) - center) / scale).astype(np.float32)

    def _scale_window_to_minus_one_plus_one(self, values: np.ndarray) -> Tuple[np.ndarray, float]:
        values = np.asarray(values, dtype=np.float32).reshape(-1)
        q_low = float(np.quantile(values, self._local_price_lower_q))
        q_high = float(np.quantile(values, self._local_price_upper_q))
        spread = q_high - q_low
        if spread < 1e-6:
            return np.zeros_like(values, dtype=np.float32), 1.0
        clipped = np.clip(values, q_low, q_high)
        scaled = 2.0 * (clipped - q_low) / spread - 1.0
        return np.clip(scaled, -1.0, 1.0).astype(np.float32), float(spread)

    def _build_price_features(self, current_price: float, price_forecast: np.ndarray) -> Tuple[float, np.ndarray]:
        window = np.concatenate([np.array([current_price], dtype=np.float32), price_forecast.astype(np.float32)])
        scaled_window, spread = self._scale_window_to_minus_one_plus_one(window)
        current_price_feature = float(scaled_window[0])

        if self._price_feature_mode == "local_delta":
            future_feature = np.clip((price_forecast - current_price) / spread, -1.0, 1.0).astype(np.float32)
        else:
            future_feature = scaled_window[1:].astype(np.float32)

        return current_price_feature, future_feature

    def _build_forecast_from_truth(self, t: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        future_slice = self._data.iloc[t + 1:t + 1 + self.FORECAST_HORIZON]

        future_load = future_slice.iloc[:, 0].to_numpy(dtype=np.float32)
        future_pv = future_slice.iloc[:, 1].to_numpy(dtype=np.float32)
        prosumption = future_load - future_pv

        price = future_slice.iloc[:, 2].to_numpy(dtype=np.float32)
        emissions = future_slice.iloc[:, 3].to_numpy(dtype=np.float32)

        if len(prosumption) < self.FORECAST_HORIZON:
            pad_len = self.FORECAST_HORIZON - len(prosumption)
            current_net = float(self._data.iloc[t, 0] - self._data.iloc[t, 1])
            current_price = float(self._data.iloc[t, 2])
            current_emission = float(self._data.iloc[t, 3])

            last_net = float(prosumption[-1]) if len(prosumption) > 0 else current_net
            last_price = float(price[-1]) if len(price) > 0 else current_price
            last_emission = float(emissions[-1]) if len(emissions) > 0 else current_emission

            prosumption = np.pad(prosumption, (0, pad_len), mode="constant", constant_values=last_net)
            price = np.pad(price, (0, pad_len), mode="constant", constant_values=last_price)
            emissions = np.pad(emissions, (0, pad_len), mode="constant", constant_values=last_emission)

        return prosumption, price, emissions

    def _build_observation(self, t: int) -> np.ndarray:
        load_kwh = float(self._data.iloc[t, 0])
        pv_kwh = float(self._data.iloc[t, 1])
        price = float(self._data.iloc[t, 2])
        emissions = float(self._data.iloc[t, 3])

        net_load_kwh = load_kwh - pv_kwh
        soc = self._soe_kwh / self._capacity_kwh if self._capacity_kwh > 0.0 else 0.0

        slot_in_day = t % 48
        hour_angle = 2.0 * np.pi * slot_in_day / 48.0
        hour_sin = np.sin(hour_angle)
        hour_cos = np.cos(hour_angle)

        if self._forecast_df is None:
            prosumption_forecast, price_forecast, emission_forecast = self._build_forecast_from_truth(t)
        else:
            row = self._forecast_df.iloc[t]
            prosumption_forecast = row[self._prosumption_cols].to_numpy(dtype=np.float32)
            price_forecast = row[self._price_cols].to_numpy(dtype=np.float32)
            emission_forecast = row[self._emission_cols].to_numpy(dtype=np.float32)

        prosumption_forecast = prosumption_forecast + self._sample_noise(self.FORECAST_HORIZON)
        price_forecast = price_forecast + self._sample_noise(self.FORECAST_HORIZON)
        emission_forecast = emission_forecast + self._sample_noise(self.FORECAST_HORIZON)

        observation_parts = [
            np.array(
                [
                    soc,
                    float(self._robust_normalize(np.array([net_load_kwh], dtype=np.float32), "net_load")[0]),
                    np.float32(hour_sin),
                    np.float32(hour_cos),
                ],
                dtype=np.float32,
            ),
            self._robust_normalize(prosumption_forecast, "net_load"),
        ]

        if self._use_price_features:
            current_price_feature, future_price_feature = self._build_price_features(price, price_forecast)
            observation_parts.insert(1, np.array([current_price_feature], dtype=np.float32))
            observation_parts.append(future_price_feature)

        if self._use_emission_features:
            current_emission_feature = self._robust_normalize(np.array([emissions], dtype=np.float32), "emission")
            future_emission_feature = self._robust_normalize(emission_forecast, "emission")

            if self._use_price_features:
                observation_parts.insert(2, current_emission_feature.astype(np.float32))
            else:
                observation_parts.insert(1, current_emission_feature.astype(np.float32))

            observation_parts.append(future_emission_feature)

        return np.concatenate(observation_parts).astype(np.float32)

    def record_decision_time(self, decision_time_seconds: float) -> None:
        if self._track_history and len(self._history) > 0:
            self._history[-1]["decision_time_seconds"] = float(decision_time_seconds)

class NoBess:
    """
    No control action.
    Battery always stays idle.
    """

    def __init__(
        self,
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
    ):
        self.ecoPriority = float(ecoPriority)
        self.feed_in_price = float(feed_in_price)
        self.power_grid_kw = float(power_grid_kw)
        self.init_charge_kwh = float(init_charge_kwh)
        self.penalty_factor = float(penalty_factor)

    def _run_building(
        self,
        building_key: str,
        data: pd.DataFrame,
        bess_meta: Dict[str, float],
        obs_stats: Dict[str, float],
    ) -> Tuple[Dict[str, Any], pd.DataFrame]:
        env = EnergyManagementEnv(
            data=data,
            bess_meta=bess_meta,
            ecoPriority=self.ecoPriority,
            feed_in_price=self.feed_in_price,
            power_grid_kw=self.power_grid_kw,
            init_charge_kwh=self.init_charge_kwh,
            penalty_factor=self.penalty_factor,
            obs_stats=obs_stats,
            track_history=True,
        )

        start_time = time.perf_counter()
        time_step = env.reset()

        while not bool(time_step.is_last()):
            decision_start = time.perf_counter()
            action = np.array([0.0], dtype=np.float32)
            decision_time = time.perf_counter() - decision_start

            time_step = env.step(action)
            env.record_decision_time(decision_time)

        runtime = time.perf_counter() - start_time
        return (
            env.summary(
                "noBess",
                building_key,
                runtime_seconds=runtime,
                training_seconds=0.0,
            ),
            env.history_df(),
        )

    def run(self, prepared: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        rows = []
        histories: Dict[str, pd.DataFrame] = {}

        for building_key, data in prepared["dataset"]["test"].items():
            summary, history_df = self._run_building(
                building_key=building_key,
                data=data,
                bess_meta=prepared["bess_meta"]["train"][building_key],
                obs_stats=prepared["obs_stats"][building_key],
            )
            rows.append(summary)
            histories[building_key] = history_df

        return pd.DataFrame(rows), histories


class RuleBess:
    """
    Simple rule based controller.

    Charge from PV surplus first.
    Discharge only against local positive demand.
    Optionally allow conservative grid charging in cheap or clean periods.
    """

    def __init__(
        self,
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        forecast_horizon: int = 48,
        charge_quantile: float = 0.20,
        discharge_quantile: float = 0.75,
        allow_grid_charging: bool = False,
        grid_charge_fraction: float = 0.25,
        end_horizon_fraction: float = 0.50,
    ):
        self.ecoPriority = float(ecoPriority)
        self.feed_in_price = float(feed_in_price)
        self.power_grid_kw = float(power_grid_kw)
        self.init_charge_kwh = float(init_charge_kwh)
        self.penalty_factor = float(penalty_factor)

        self.forecast_horizon = int(forecast_horizon)
        self.charge_quantile = float(charge_quantile)
        self.discharge_quantile = float(discharge_quantile)
        self.allow_grid_charging = bool(allow_grid_charging)
        self.grid_charge_fraction = float(grid_charge_fraction)
        self.end_horizon_fraction = float(end_horizon_fraction)

    @staticmethod
    def _robust_normalize(values: np.ndarray) -> np.ndarray:
        values = np.asarray(values, dtype=np.float64).reshape(-1)
        q_low = float(np.quantile(values, 0.01))
        q_high = float(np.quantile(values, 0.99))
        clipped = np.clip(values, q_low, q_high)
        center = float(np.median(clipped))
        scale = float(np.quantile(clipped, 0.75) - np.quantile(clipped, 0.25))
        if scale < 1e-9:
            scale = float(np.std(clipped))
        if scale < 1e-9:
            scale = 1.0
        return (clipped - center) / scale

    def _choose_action(
        self,
        t: int,
        env: EnergyManagementEnv,
        net_load: np.ndarray,
        combined_signal: np.ndarray,
    ) -> float:
        current_signal = float(combined_signal[t])
        current_net_load = float(net_load[t])

        end_idx = min(len(net_load), t + 1 + self.forecast_horizon)
        signal_window = combined_signal[t:end_idx]
        if len(signal_window) == 0:
            signal_window = np.array([current_signal], dtype=np.float64)

        low_signal = float(np.quantile(signal_window, self.charge_quantile))
        high_signal = float(np.quantile(signal_window, self.discharge_quantile))
        median_signal = float(np.quantile(signal_window, 0.50))

        max_step_energy_kwh = env.power_battery_kw * 0.5
        charge_headroom_kwh = max(env.capacity_kwh - env.soe_kwh, 0.0)
        discharge_headroom_kwh = max(env.soe_kwh, 0.0)

        feasible_charge_kwh = min(max_step_energy_kwh, charge_headroom_kwh)
        feasible_discharge_kwh = min(max_step_energy_kwh, discharge_headroom_kwh)

        remaining_steps = len(net_load) - 1 - t
        end_horizon_steps = max(1, int(self.forecast_horizon * self.end_horizon_fraction))

        if current_net_load < 0.0 and feasible_charge_kwh > 1e-9:
            requested_delta_soe_kwh = min(-current_net_load, feasible_charge_kwh)

        elif current_net_load > 0.0 and feasible_discharge_kwh > 1e-9:
            local_discharge_limit = min(current_net_load, feasible_discharge_kwh)

            if remaining_steps <= end_horizon_steps:
                requested_delta_soe_kwh = -local_discharge_limit if current_signal >= median_signal else 0.0
            else:
                requested_delta_soe_kwh = -local_discharge_limit if current_signal >= high_signal else 0.0

        elif self.allow_grid_charging and feasible_charge_kwh > 1e-9 and current_signal <= low_signal:
            future_positive_load = np.maximum(net_load[t:end_idx], 0.0).sum()
            useful_additional_storage = max(float(future_positive_load) - env.soe_kwh, 0.0)

            requested_delta_soe_kwh = min(
                self.grid_charge_fraction * max_step_energy_kwh,
                feasible_charge_kwh,
                useful_additional_storage,
            )
        else:
            requested_delta_soe_kwh = 0.0

        return float(np.clip(requested_delta_soe_kwh / max_step_energy_kwh, -1.0, 1.0))

    def _run_building(
        self,
        building_key: str,
        data: pd.DataFrame,
        bess_meta: Dict[str, float],
        obs_stats: Dict[str, float],
    ) -> Tuple[Dict[str, Any], pd.DataFrame]:
        env = EnergyManagementEnv(
            data=data,
            bess_meta=bess_meta,
            ecoPriority=self.ecoPriority,
            feed_in_price=self.feed_in_price,
            power_grid_kw=self.power_grid_kw,
            init_charge_kwh=self.init_charge_kwh,
            penalty_factor=self.penalty_factor,
            obs_stats=obs_stats,
            track_history=True,
        )

        load_kwh = data.iloc[:, 0].to_numpy(dtype=np.float64)
        pv_kwh = data.iloc[:, 1].to_numpy(dtype=np.float64)
        price = data.iloc[:, 2].to_numpy(dtype=np.float64)
        emissions = data.iloc[:, 3].to_numpy(dtype=np.float64)
        net_load = load_kwh - pv_kwh

        price_signal = self._robust_normalize(price)
        emission_signal = self._robust_normalize(emissions)
        combined_signal = (1.0 - self.ecoPriority) * price_signal + self.ecoPriority * emission_signal

        start_time = time.perf_counter()
        time_step = env.reset()
        t = 0

        while not bool(time_step.is_last()):
            decision_start = time.perf_counter()
            action_norm = self._choose_action(t, env, net_load, combined_signal)
            decision_time = time.perf_counter() - decision_start

            time_step = env.step(np.array([action_norm], dtype=np.float32))
            env.record_decision_time(decision_time)
            t += 1

        runtime = time.perf_counter() - start_time
        return (
            env.summary(
                "RuleBess",
                building_key,
                runtime_seconds=runtime,
                training_seconds=0.0,
            ),
            env.history_df(),
        )

    def run(self, prepared: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        rows = []
        histories: Dict[str, pd.DataFrame] = {}

        for building_key, data in prepared["dataset"]["test"].items():
            summary, history_df = self._run_building(
                building_key=building_key,
                data=data,
                bess_meta=prepared["bess_meta"]["train"][building_key],
                obs_stats=prepared["obs_stats"][building_key],
            )
            rows.append(summary)
            histories[building_key] = history_df

        return pd.DataFrame(rows), histories


class MIP:
    """
    Perfect foresight full horizon LP.
    The environment is only used for replay and consistent accounting.
    """

    def __init__(
        self,
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        solver_name: Optional[str] = None,
        enforce_terminal_soe: bool = False,
    ):
        self.ecoPriority = float(ecoPriority)
        self.feed_in_price = float(feed_in_price)
        self.power_grid_kw = float(power_grid_kw)
        self.init_charge_kwh = float(init_charge_kwh)
        self.penalty_factor = float(penalty_factor)
        self.solver_name = solver_name
        self.enforce_terminal_soe = bool(enforce_terminal_soe)

    def _select_solver(self) -> str:
        if self.solver_name is not None:
            solver = pyo.SolverFactory(self.solver_name)
            if solver is None or not solver.available(exception_flag=False):
                raise ValueError(f"Requested solver '{self.solver_name}' is not available.")
            return self.solver_name

        for name in ["gurobi", "appsi_highs", "highs", "cbc", "glpk"]:
            solver = pyo.SolverFactory(name)
            if solver is not None and solver.available(exception_flag=False):
                return name

        raise RuntimeError("No LP solver found. Install gurobi or highspy.")

    def _solve_action_sequence(
        self,
        net_load_kwh: np.ndarray,
        price: np.ndarray,
        emissions: np.ndarray,
        capacity_kwh: float,
        power_battery_kw: float,
        init_charge_kwh: float,
        enforce_terminal_soe: bool,
    ) -> np.ndarray:
        horizon = len(net_load_kwh)

        m = pyo.ConcreteModel()
        m.T = pyo.RangeSet(0, horizon - 1)
        m.S = pyo.RangeSet(0, horizon)

        m.action = pyo.Var(m.T, bounds=(-1.0, 1.0))
        m.soe = pyo.Var(m.S, bounds=(0.0, capacity_kwh))
        m.grid_buy = pyo.Var(m.T, domain=pyo.NonNegativeReals)
        m.grid_sell = pyo.Var(m.T, domain=pyo.NonNegativeReals)

        m.init_soe = pyo.Constraint(expr=m.soe[0] == init_charge_kwh)

        def soe_rule(model, t):
            return model.soe[t + 1] == model.soe[t] + 0.5 * power_battery_kw * model.action[t]

        m.soe_dyn = pyo.Constraint(m.T, rule=soe_rule)

        def balance_rule(model, t):
            return model.grid_buy[t] - model.grid_sell[t] == net_load_kwh[t] + 0.5 * power_battery_kw * model.action[t]

        m.balance = pyo.Constraint(m.T, rule=balance_rule)

        if enforce_terminal_soe:
            m.terminal_soe = pyo.Constraint(expr=m.soe[horizon] == init_charge_kwh)

        def objective_rule(model):
            return sum(
                (1.0 - self.ecoPriority) * (model.grid_sell[t] * self.feed_in_price - model.grid_buy[t] * price[t])
                - self.ecoPriority * model.grid_buy[t] * emissions[t]
                for t in model.T
            )

        m.obj = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

        solver = pyo.SolverFactory(self._select_solver())
        results = solver.solve(m, tee=False)

        if not pyo.check_optimal_termination(results):
            raise RuntimeError(
                f"MIP did not terminate optimally. "
                f"Status: {results.solver.status}, "
                f"Termination: {results.solver.termination_condition}"
            )

        return np.array([float(pyo.value(m.action[t])) for t in m.T], dtype=np.float32)

    def _run_building(
        self,
        building_key: str,
        data: pd.DataFrame,
        bess_meta: Dict[str, float],
        obs_stats: Dict[str, float],
    ) -> Tuple[Dict[str, Any], pd.DataFrame]:
        env = EnergyManagementEnv(
            data=data,
            bess_meta=bess_meta,
            ecoPriority=self.ecoPriority,
            feed_in_price=self.feed_in_price,
            power_grid_kw=self.power_grid_kw,
            init_charge_kwh=self.init_charge_kwh,
            penalty_factor=self.penalty_factor,
            obs_stats=obs_stats,
            track_history=True,
        )

        net_load_kwh = data.iloc[:, 0].to_numpy(dtype=np.float64) - data.iloc[:, 1].to_numpy(dtype=np.float64)
        price = data.iloc[:, 2].to_numpy(dtype=np.float64)
        emissions = data.iloc[:, 3].to_numpy(dtype=np.float64)

        start_time = time.perf_counter()

        solve_start = time.perf_counter()
        actions = self._solve_action_sequence(
            net_load_kwh=net_load_kwh,
            price=price,
            emissions=emissions,
            capacity_kwh=float(bess_meta["capacity_kwh"]),
            power_battery_kw=float(bess_meta["power_battery_kw"]),
            init_charge_kwh=self.init_charge_kwh,
            enforce_terminal_soe=self.enforce_terminal_soe,
        )
        solve_time = time.perf_counter() - solve_start

        per_action_decision_time = solve_time / max(len(actions), 1)

        time_step = env.reset()
        for action in actions:
            if bool(time_step.is_last()):
                break
            time_step = env.step(np.array([action], dtype=np.float32))
            env.record_decision_time(per_action_decision_time)

        runtime = time.perf_counter() - start_time
        return (
            env.summary(
                "MIP",
                building_key,
                runtime_seconds=runtime,
                training_seconds=0.0,
                extra_metrics={"OptimizationSolveSeconds": float(solve_time)},
            ),
            env.history_df(),
        )
    
    def run(self, prepared: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        rows = []
        histories: Dict[str, pd.DataFrame] = {}

        for building_key, data in prepared["dataset"]["test"].items():
            summary, history_df = self._run_building(
                building_key=building_key,
                data=data,
                bess_meta=prepared["bess_meta"]["train"][building_key],
                obs_stats=prepared["obs_stats"][building_key],
            )
            rows.append(summary)
            histories[building_key] = history_df

        return pd.DataFrame(rows), histories


class MPC(MIP):
    """
    Receding horizon control.
    Uses the same LP core as MIP, but only applies the first action of each solve.
    """

    def __init__(
        self,
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        solver_name: Optional[str] = None,
        forecast_horizon: int = 48,
        terminal_soe_constraint: bool = False,
    ):
        super().__init__(
            ecoPriority=ecoPriority,
            feed_in_price=feed_in_price,
            power_grid_kw=power_grid_kw,
            init_charge_kwh=init_charge_kwh,
            penalty_factor=penalty_factor,
            solver_name=solver_name,
            enforce_terminal_soe=False,
        )
        self.forecast_horizon = int(forecast_horizon)
        self.terminal_soe_constraint = bool(terminal_soe_constraint)

    def _get_planning_vectors(
        self,
        t: int,
        data: pd.DataFrame,
        forecast_df: Optional[pd.DataFrame],
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        current_net = np.array([float(data.iloc[t, 0] - data.iloc[t, 1])], dtype=np.float64)
        current_price = np.array([float(data.iloc[t, 2])], dtype=np.float64)
        current_emission = np.array([float(data.iloc[t, 3])], dtype=np.float64)

        if forecast_df is None:
            end_idx = min(len(data), t + 1 + self.forecast_horizon)

            future_net = (
                data.iloc[t + 1:end_idx, 0].to_numpy(dtype=np.float64)
                - data.iloc[t + 1:end_idx, 1].to_numpy(dtype=np.float64)
            )
            future_price = data.iloc[t + 1:end_idx, 2].to_numpy(dtype=np.float64)
            future_emission = data.iloc[t + 1:end_idx, 3].to_numpy(dtype=np.float64)

            if len(future_net) < self.forecast_horizon:
                pad_len = self.forecast_horizon - len(future_net)
                last_net = float(future_net[-1]) if len(future_net) > 0 else float(current_net[0])
                last_price = float(future_price[-1]) if len(future_price) > 0 else float(current_price[0])
                last_emission = float(future_emission[-1]) if len(future_emission) > 0 else float(current_emission[0])

                future_net = np.pad(future_net, (0, pad_len), mode="constant", constant_values=last_net)
                future_price = np.pad(future_price, (0, pad_len), mode="constant", constant_values=last_price)
                future_emission = np.pad(future_emission, (0, pad_len), mode="constant", constant_values=last_emission)
        else:
            row = forecast_df.iloc[t]
            future_net = row[[f"prosumption_f{i}" for i in range(1, self.forecast_horizon + 1)]].to_numpy(dtype=np.float64)
            future_price = row[[f"price_f{i}" for i in range(1, self.forecast_horizon + 1)]].to_numpy(dtype=np.float64)
            future_emission = row[[f"emission_f{i}" for i in range(1, self.forecast_horizon + 1)]].to_numpy(dtype=np.float64)

        return (
            np.concatenate([current_net, future_net]),
            np.concatenate([current_price, future_price]),
            np.concatenate([current_emission, future_emission]),
        )

    def _run_building(
        self,
        building_key: str,
        data: pd.DataFrame,
        bess_meta: Dict[str, float],
        obs_stats: Dict[str, float],
        forecast_df: Optional[pd.DataFrame],
    ) -> Tuple[Dict[str, Any], pd.DataFrame]:
        env = EnergyManagementEnv(
            data=data,
            bess_meta=bess_meta,
            ecoPriority=self.ecoPriority,
            feed_in_price=self.feed_in_price,
            power_grid_kw=self.power_grid_kw,
            init_charge_kwh=self.init_charge_kwh,
            penalty_factor=self.penalty_factor,
            forecast_df=forecast_df,
            obs_stats=obs_stats,
            track_history=True,
        )

        start_time = time.perf_counter()
        time_step = env.reset()
        t = 0

        while not bool(time_step.is_last()):
            decision_start = time.perf_counter()

            plan_net, plan_price, plan_emission = self._get_planning_vectors(t, data, forecast_df)

            first_action = self._solve_action_sequence(
                net_load_kwh=plan_net,
                price=plan_price,
                emissions=plan_emission,
                capacity_kwh=float(bess_meta["capacity_kwh"]),
                power_battery_kw=float(bess_meta["power_battery_kw"]),
                init_charge_kwh=env.soe_kwh,
                enforce_terminal_soe=self.terminal_soe_constraint,
            )[0]

            decision_time = time.perf_counter() - decision_start

            time_step = env.step(np.array([first_action], dtype=np.float32))
            env.record_decision_time(decision_time)
            t += 1

        runtime = time.perf_counter() - start_time
        return (
            env.summary(
                "MPC",
                building_key,
                runtime_seconds=runtime,
                training_seconds=0.0,
            ),
            env.history_df(),
        )

    def run(self, prepared: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        rows = []
        histories: Dict[str, pd.DataFrame] = {}

        for building_key, data in prepared["dataset"]["test"].items():
            forecast_df = None
            if prepared["forecast_tables"] is not None:
                forecast_df = prepared["forecast_tables"]["test"][building_key]

            summary, history_df = self._run_building(
                building_key=building_key,
                data=data,
                bess_meta=prepared["bess_meta"]["train"][building_key],
                obs_stats=prepared["obs_stats"][building_key],
                forecast_df=forecast_df,
            )
            rows.append(summary)
            histories[building_key] = history_df

        return pd.DataFrame(rows), histories


class LL_RL:
    """
    Local DDPG.
    One model per building.
    """

    def __init__(
        self,
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        noise_std: float = 0.0,
        base_seed: int = 42,
        batch_size: int = 128,
        replay_buffer_capacity: int = 200000,
        initial_collect_steps: int = 10000,
        collect_steps_per_iteration: int = 1,
        num_iterations: int = 50000,
        actor_learning_rate: float = 1e-4,
        critic_learning_rate: float = 3e-4,
        price_feature_mode: str = "local_absolute",
        local_price_lower_q: float = 0.05,
        local_price_upper_q: float = 0.95,
        algorithm: str = "DDPG",
        ppo_collect_steps: int = 2048,   # kept only so RunExperiments can still pass it
        ppo_num_epochs: int = 10,        # kept only so RunExperiments can still pass it
    ):
        self.ecoPriority = float(ecoPriority)
        self.feed_in_price = float(feed_in_price)
        self.power_grid_kw = float(power_grid_kw)
        self.init_charge_kwh = float(init_charge_kwh)
        self.penalty_factor = float(penalty_factor)

        self.noise_std = float(noise_std)
        self.base_seed = int(base_seed)
        self.batch_size = int(batch_size)
        self.replay_buffer_capacity = int(replay_buffer_capacity)
        self.initial_collect_steps = int(initial_collect_steps)
        self.collect_steps_per_iteration = int(collect_steps_per_iteration)
        self.num_iterations = int(num_iterations)

        self.actor_learning_rate = float(actor_learning_rate)
        self.critic_learning_rate = float(critic_learning_rate)

        self.price_feature_mode = str(price_feature_mode)
        self.local_price_lower_q = float(local_price_lower_q)
        self.local_price_upper_q = float(local_price_upper_q)

        self.algorithm = str(algorithm).upper()
        self.ppo_collect_steps = 1024
        self.ppo_num_epochs = 10
        self.ppo_local_updates = 30
        self.ppo_local_updates_per_round = 5
        self.ppo_local_retraining_updates = 1

        valid_algorithms = {"DDPG", "TD3", "SAC", "PPO"}
        if self.algorithm not in valid_algorithms:
            raise ValueError(f"algorithm must be one of {valid_algorithms}, got {self.algorithm}")

    def _set_all_seeds(self, seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)

    def _build_tf_env(
        self,
        data: pd.DataFrame,
        bess_meta: Dict[str, float],
        obs_stats: Dict[str, float],
        forecast_df: Optional[pd.DataFrame],
        track_history: bool,
        seed: int,
    ) -> tf_py_environment.TFPyEnvironment:
        return tf_py_environment.TFPyEnvironment(
            EnergyManagementEnv(
                data=data,
                bess_meta=bess_meta,
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
                forecast_df=forecast_df,
                obs_stats=obs_stats,
                track_history=track_history,
                noise_std=self.noise_std,
                random_seed=seed,
                price_feature_mode=self.price_feature_mode,
                local_price_lower_q=self.local_price_lower_q,
                local_price_upper_q=self.local_price_upper_q,
            )
        )

    def _create_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        if self.algorithm == "DDPG":
            return self._create_ddpg_agent(train_env)
        elif self.algorithm == "TD3":
            return self._create_td3_agent(train_env)
        elif self.algorithm == "SAC":
            return self._create_sac_agent(train_env)
        elif self.algorithm == "PPO":
            return self._create_ppo_agent(train_env)
        else:
            raise ValueError(f"Unsupported algorithm: {self.algorithm}")
        
    def _create_ddpg_agent(self, train_env: tf_py_environment.TFPyEnvironment) -> ddpg_agent.DdpgAgent:
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(400, 300),
            activation_fn=tf.keras.activations.relu,
        )

        critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(400,),
            joint_fc_layer_params=(300,),
            activation_fn=tf.keras.activations.relu,
        )

        target_actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(400, 300),
            activation_fn=tf.keras.activations.relu,
        )

        target_critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(400,),
            joint_fc_layer_params=(300,),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = ddpg_agent.DdpgAgent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            actor_network=actor_net,
            critic_network=critic_net,
            target_actor_network=target_actor_net,
            target_critic_network=target_critic_net,
            actor_optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            critic_optimizer=tf.keras.optimizers.Adam(self.critic_learning_rate),
            ou_stddev=0.3,
            ou_damping=0.1,
            target_update_tau=0.001,
            target_update_period=1,
            td_errors_loss_fn=common.element_wise_huber_loss,
            gamma=0.99999,
            reward_scale_factor=1.0,
            train_step_counter=train_step_counter,
        )
        agent.initialize()
        return agent

    def _create_td3_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256,),
            activation_fn=tf.keras.activations.relu,
        )

        target_actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        target_critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256,),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = td3_agent.Td3Agent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            actor_network=actor_net,
            critic_network=critic_net,
            target_actor_network=target_actor_net,
            target_critic_network=target_critic_net,
            actor_optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            critic_optimizer=tf.keras.optimizers.Adam(self.critic_learning_rate),
            target_update_tau=0.005,
            target_update_period=1,
            exploration_noise_std=0.1,
            target_policy_noise=0.2,
            target_policy_noise_clip=0.5,
            gamma=0.99999,
            reward_scale_factor=1.0,
            train_step_counter=train_step_counter,
        )
        agent.initialize()
        return agent

    def _create_sac_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_distribution_network.ActorDistributionNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            continuous_projection_net=tanh_normal_projection_network.TanhNormalProjectionNetwork,
            activation_fn=tf.keras.activations.relu,
        )

        critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = sac_agent.SacAgent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            actor_network=actor_net,
            critic_network=critic_net,
            actor_optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            critic_optimizer=tf.keras.optimizers.Adam(self.critic_learning_rate),
            alpha_optimizer=tf.keras.optimizers.Adam(3e-4),
            target_update_tau=0.005,
            target_update_period=1,
            td_errors_loss_fn=common.element_wise_huber_loss,
            gamma=0.99999,
            reward_scale_factor=1.0,
            train_step_counter=train_step_counter,
        )
        agent.initialize()
        return agent

    def _create_ppo_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_distribution_network.ActorDistributionNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        value_net = value_network.ValueNetwork(
            input_tensor_spec=obs_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = ppo_clip_agent.PPOClipAgent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            actor_net=actor_net,
            value_net=value_net,
            importance_ratio_clipping=0.2,
            discount_factor=0.99999,
            entropy_regularization=0.0,
            value_pred_loss_coef=0.5,
            gradient_clipping=0.5,
            use_gae=True,
            lambda_value=0.95,
            num_epochs=self.ppo_num_epochs,
            train_step_counter=train_step_counter,
        )
        agent.initialize()
        return agent

    def _create_off_policy_pipeline(
        self,
        agent: ddpg_agent.DdpgAgent,
        train_env: tf_py_environment.TFPyEnvironment,
    ):
        replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
            data_spec=agent.collect_data_spec,
            batch_size=train_env.batch_size,
            max_length=self.replay_buffer_capacity,
        )

        random_policy = random_tf_policy.RandomTFPolicy(
            time_step_spec=train_env.time_step_spec(),
            action_spec=train_env.action_spec(),
        )

        initial_collect_driver = dynamic_step_driver.DynamicStepDriver(
            env=train_env,
            policy=random_policy,
            observers=[replay_buffer.add_batch],
            num_steps=self.initial_collect_steps,
        )

        collect_driver = dynamic_step_driver.DynamicStepDriver(
            env=train_env,
            policy=agent.collect_policy,
            observers=[replay_buffer.add_batch],
            num_steps=self.collect_steps_per_iteration,
        )

        initial_collect_driver.run = common.function(initial_collect_driver.run)
        collect_driver.run = common.function(collect_driver.run)
        agent.train = common.function(agent.train)

        initial_collect_driver.run(
            time_step=train_env.reset(),
            policy_state=random_policy.get_initial_state(train_env.batch_size),
        )

        dataset = replay_buffer.as_dataset(
            num_parallel_calls=tf.data.AUTOTUNE,
            sample_batch_size=self.batch_size,
            num_steps=2,
        ).prefetch(tf.data.AUTOTUNE)

        iterator = iter(dataset)
        time_step = train_env.reset()
        policy_state = agent.collect_policy.get_initial_state(train_env.batch_size)

        return collect_driver, iterator, time_step, policy_state

    def _create_ppo_pipeline(
        self,
        agent,
        train_env: tf_py_environment.TFPyEnvironment,
    ):
        replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
            data_spec=agent.collect_data_spec,
            batch_size=train_env.batch_size,
            max_length=self.ppo_collect_steps + 1,
        )

        collect_driver = dynamic_step_driver.DynamicStepDriver(
            env=train_env,
            policy=agent.collect_policy,
            observers=[replay_buffer.add_batch],
            num_steps=self.ppo_collect_steps,
        )

        collect_driver.run = common.function(collect_driver.run)
        agent.train = common.function(agent.train)

        return replay_buffer, collect_driver

    def _unwrap_py_env(self, tf_env: tf_py_environment.TFPyEnvironment):
        py_env = tf_env.pyenv
        while hasattr(py_env, "envs") and len(py_env.envs) == 1:
            py_env = py_env.envs[0]
        return py_env

    def _evaluate_policy(self, tf_env: tf_py_environment.TFPyEnvironment, policy) -> Dict[str, Any]:
        py_env = self._unwrap_py_env(tf_env)
        time_step = tf_env.reset()
        policy_state = policy.get_initial_state(tf_env.batch_size)

        evaluation_start = time.perf_counter()

        while not bool(time_step.is_last()):
            decision_start = time.perf_counter()
            action_step = policy.action(time_step, policy_state)
            decision_time = time.perf_counter() - decision_start

            policy_state = action_step.state
            time_step = tf_env.step(action_step.action)
            py_env.record_decision_time(decision_time)

        evaluation_runtime_seconds = time.perf_counter() - evaluation_start

        return {
            "PyEnv": py_env,
            "Total Profit": py_env.total_profit,
            "Total Emissions": py_env.total_emissions,
            "History": py_env.history_df(),
            "EvaluationRuntimeSeconds": float(evaluation_runtime_seconds),
        }
    
    def _train_one_building(
        self,
        building_key: str,
        prepared: Dict[str, Any],
        seed: int,
    ) -> Tuple[Dict[str, Any], pd.DataFrame]:
        self._set_all_seeds(seed)

        train_forecast = prepared["forecast_tables"]["train"][building_key] if prepared["forecast_tables"] is not None else None
        eval_forecast = prepared["forecast_tables"]["eval"][building_key] if prepared["forecast_tables"] is not None else None
        test_forecast = prepared["forecast_tables"]["test"][building_key] if prepared["forecast_tables"] is not None else None

        train_env = self._build_tf_env(
            data=prepared["dataset"]["train"][building_key],
            bess_meta=prepared["bess_meta"]["train"][building_key],
            obs_stats=prepared["obs_stats"][building_key],
            forecast_df=train_forecast,
            track_history=False,
            seed=seed,
        )
        eval_env = self._build_tf_env(
            data=prepared["dataset"]["eval"][building_key],
            bess_meta=prepared["bess_meta"]["train"][building_key],
            obs_stats=prepared["obs_stats"][building_key],
            forecast_df=eval_forecast,
            track_history=False,
            seed=seed,
        )
        test_env = self._build_tf_env(
            data=prepared["dataset"]["test"][building_key],
            bess_meta=prepared["bess_meta"]["train"][building_key],
            obs_stats=prepared["obs_stats"][building_key],
            forecast_df=test_forecast,
            track_history=True,
            seed=seed,
        )

        _ = eval_env

        agent = self._create_agent(train_env)

        training_start = time.perf_counter()

        if self.algorithm in {"DDPG", "TD3", "SAC"}:
            collect_driver, iterator, time_step, policy_state = self._create_off_policy_pipeline(agent, train_env)

            while int(agent.train_step_counter.numpy()) < self.num_iterations:
                time_step, policy_state = collect_driver.run(
                    time_step=time_step,
                    policy_state=policy_state,
                )
                experience, _ = next(iterator)
                agent.train(experience)

        elif self.algorithm == "PPO":
            replay_buffer, collect_driver = self._create_ppo_pipeline(agent, train_env)

            for _ in range(self.ppo_local_updates):
                replay_buffer.clear()
                collect_driver.run(
                    time_step=train_env.reset(),
                    policy_state=agent.collect_policy.get_initial_state(train_env.batch_size),
                )

                experience = replay_buffer.gather_all()
                agent.train(experience)

        else:
            raise ValueError(f"Unsupported algorithm: {self.algorithm}")

        training_seconds = time.perf_counter() - training_start

        eval_result = self._evaluate_policy(test_env, agent.policy)
        py_env = eval_result["PyEnv"]

        runtime_seconds = training_seconds + float(eval_result["EvaluationRuntimeSeconds"])

        summary = py_env.summary(
            agent_name=f"LL_{self.algorithm}",
            building_key=building_key,
            runtime_seconds=runtime_seconds,
            training_seconds=training_seconds,
            extra_metrics={
                "Seed": int(seed),
                "TrainSteps": int(agent.train_step_counter.numpy()),
            },
        )

        return summary, eval_result["History"]

    def run(self, prepared: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        rows = []
        histories: Dict[str, pd.DataFrame] = {}

        for building_idx, building_key in enumerate(prepared["dataset"]["train"].keys(), start=1):
            seed = self.base_seed + 1000 * building_idx
            summary, history_df = self._train_one_building(building_key, prepared, seed)
            rows.append(summary)
            histories[building_key] = history_df

        return pd.DataFrame(rows), histories


class FL_RL(LL_RL):
    """
    Federated RL for DDPG, TD3, SAC and PPO.

    Features
    --------
    - optional clustered aggregation using the provided cluster file
    - one aggregation function only
    - optional clipping
    - optional differential privacy noise
    - optional limited local retraining after FL rounds

    Important
    ---------
    This class is written as a drop-in replacement and therefore redefines
    the agent creation methods so that the federated networks are stored
    explicitly in `agent._fed_networks`.
    """

    def __init__(
        self,
        ecoPriority: float,
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        noise_std: float = 0.0,
        base_seed: int = 42,
        batch_size: int = 128,
        replay_buffer_capacity: int = 200000,
        initial_collect_steps: int = 10000,
        collect_steps_per_iteration: int = 1,
        num_iterations: int = 50000,
        actor_learning_rate: float = 1e-4,
        critic_learning_rate: float = 3e-4,
        price_feature_mode: str = "local_absolute",
        local_price_lower_q: float = 0.05,
        local_price_upper_q: float = 0.95,
        algorithm: str = "DDPG",
        ppo_collect_steps: int = 2048,   # kept only so RunExperiments can still pass it
        ppo_num_epochs: int = 10,        # kept only so RunExperiments can still pass it
        fed_rounds: int = 10,
        local_steps_per_round: int = 5000,
        clustered_aggregation: bool = True,
        cluster_file_path: Optional[str] = None,
        clipping_enabled: bool = False,
        clip_norm: float = 1.0,
        differential_privacy: bool = False,
        dp_noise_multiplier: float = 0.0,
        local_retraining_steps: int = 0,
        weighted_aggregation: bool = False,
        performance_weight_alpha: float = 0.5,
        num_clusters: int = 1,
        cluster_feature_names: Optional[List[str]] = None,
    ):
        super().__init__(
            ecoPriority=ecoPriority,
            feed_in_price=feed_in_price,
            power_grid_kw=power_grid_kw,
            init_charge_kwh=init_charge_kwh,
            penalty_factor=penalty_factor,
            noise_std=noise_std,
            base_seed=base_seed,
            batch_size=batch_size,
            replay_buffer_capacity=replay_buffer_capacity,
            initial_collect_steps=initial_collect_steps,
            collect_steps_per_iteration=collect_steps_per_iteration,
            num_iterations=num_iterations,
            actor_learning_rate=actor_learning_rate,
            critic_learning_rate=critic_learning_rate,
            price_feature_mode=price_feature_mode,
            local_price_lower_q=local_price_lower_q,
            local_price_upper_q=local_price_upper_q,
            algorithm=algorithm,
            ppo_collect_steps=ppo_collect_steps,
            ppo_num_epochs=ppo_num_epochs,
        )

        self.fed_rounds = int(fed_rounds)
        self.local_steps_per_round = int(local_steps_per_round)

        self.clustered_aggregation = bool(clustered_aggregation)
        self.cluster_file_path = cluster_file_path

        self.clipping_enabled = bool(clipping_enabled)
        self.clip_norm = float(max(clip_norm, 1e-8))

        self.differential_privacy = bool(differential_privacy)
        self.dp_noise_multiplier = float(max(dp_noise_multiplier, 0.0))

        self.local_retraining_steps = int(max(local_retraining_steps, 0))

        self.weighted_aggregation = bool(weighted_aggregation)
        self.performance_weight_alpha = float(performance_weight_alpha)
        self.num_clusters = int(num_clusters)
        self.cluster_feature_names = cluster_feature_names

    def _create_ddpg_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256,),
            activation_fn=tf.keras.activations.relu,
        )

        target_actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        target_critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256,),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = ddpg_agent.DdpgAgent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            actor_network=actor_net,
            critic_network=critic_net,
            target_actor_network=target_actor_net,
            target_critic_network=target_critic_net,
            actor_optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            critic_optimizer=tf.keras.optimizers.Adam(self.critic_learning_rate),
            ou_stddev=0.3,
            ou_damping=0.1,
            target_update_tau=0.001,
            target_update_period=1,
            td_errors_loss_fn=common.element_wise_huber_loss,
            gamma=0.99,
            reward_scale_factor=1.0,
            train_step_counter=train_step_counter,
        )
        agent.initialize()

        agent._fed_networks = {
            "actor": actor_net,
            "critic": critic_net,
            "target_actor": target_actor_net,
            "target_critic": target_critic_net,
        }
        return agent

    def _create_td3_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256,),
            activation_fn=tf.keras.activations.relu,
        )

        target_actor_net = actor_network.ActorNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        target_critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256,),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = td3_agent.Td3Agent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            actor_network=actor_net,
            critic_network=critic_net,
            target_actor_network=target_actor_net,
            target_critic_network=target_critic_net,
            actor_optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            critic_optimizer=tf.keras.optimizers.Adam(self.critic_learning_rate),
            target_update_tau=0.005,
            target_update_period=1,
            exploration_noise_std=0.1,
            target_policy_noise=0.2,
            target_policy_noise_clip=0.5,
            gamma=0.99,
            reward_scale_factor=1.0,
            train_step_counter=train_step_counter,
        )
        agent.initialize()

        agent._fed_networks = {
            "actor": actor_net,
            "critic": critic_net,
            "target_actor": target_actor_net,
            "target_critic": target_critic_net,
        }
        return agent

    def _create_sac_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_distribution_network.ActorDistributionNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            continuous_projection_net=tanh_normal_projection_network.TanhNormalProjectionNetwork,
            activation_fn=tf.keras.activations.relu,
        )

        critic_net = critic_network.CriticNetwork(
            input_tensor_spec=(obs_spec, action_spec),
            observation_fc_layer_params=(256,),
            joint_fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = sac_agent.SacAgent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            actor_network=actor_net,
            critic_network=critic_net,
            actor_optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            critic_optimizer=tf.keras.optimizers.Adam(self.critic_learning_rate),
            alpha_optimizer=tf.keras.optimizers.Adam(3e-4),
            target_update_tau=0.005,
            target_update_period=1,
            td_errors_loss_fn=common.element_wise_huber_loss,
            gamma=0.99,
            reward_scale_factor=1.0,
            train_step_counter=train_step_counter,
        )
        agent.initialize()

        agent._fed_networks = {
            "actor": actor_net,
            "critic": critic_net,
        }
        return agent

    def _create_ppo_agent(self, train_env: tf_py_environment.TFPyEnvironment):
        obs_spec = train_env.observation_spec()
        action_spec = train_env.action_spec()

        actor_net = actor_distribution_network.ActorDistributionNetwork(
            input_tensor_spec=obs_spec,
            output_tensor_spec=action_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        value_net = value_network.ValueNetwork(
            input_tensor_spec=obs_spec,
            fc_layer_params=(256, 256),
            activation_fn=tf.keras.activations.relu,
        )

        train_step_counter = tf.Variable(0, dtype=tf.int64)

        agent = ppo_clip_agent.PPOClipAgent(
            time_step_spec=train_env.time_step_spec(),
            action_spec=action_spec,
            optimizer=tf.keras.optimizers.Adam(self.actor_learning_rate),
            actor_net=actor_net,
            value_net=value_net,
            importance_ratio_clipping=0.2,
            discount_factor=0.99,
            entropy_regularization=0.0,
            value_pred_loss_coef=0.5,
            gradient_clipping=0.5,
            use_gae=True,
            lambda_value=0.95,
            num_epochs=self.ppo_num_epochs,
            train_step_counter=train_step_counter,
        )
        agent.initialize()

        agent._fed_networks = {
            "actor": actor_net,
            "value": value_net,
        }
        return agent

    def _load_cluster_assignments(self, building_keys: List[str]) -> Dict[str, int]:
        """
        Load cluster assignments from the exact file path provided by the user.

        Expected columns:
        - dataset
        - series
        - cluster

        If clustered aggregation is disabled, all buildings are assigned to cluster 0.
        """
        if not self.clustered_aggregation:
            return {building_key: 0 for building_key in building_keys}

        if self.cluster_file_path is None:
            raise ValueError("cluster_file_path must be provided when clustered_aggregation=True.")

        if not os.path.exists(self.cluster_file_path):
            raise FileNotFoundError(f"Cluster file not found: {self.cluster_file_path}")

        cluster_df = pd.read_csv(self.cluster_file_path, sep=None, engine="python")

        required_cols = {"dataset", "series", "cluster"}
        if not required_cols.issubset(cluster_df.columns):
            raise ValueError(
                f"Cluster file must contain columns {required_cols}, "
                f"but found {list(cluster_df.columns)}"
            )

        cluster_df = cluster_df[cluster_df["dataset"].astype(str) == "Ausgrid"].copy()

        assignments = {
            str(row["series"]): int(row["cluster"])
            for _, row in cluster_df.iterrows()
        }

        missing_buildings = [building_key for building_key in building_keys if building_key not in assignments]
        if missing_buildings:
            raise ValueError(f"Missing cluster assignments for buildings: {missing_buildings}")

        return {building_key: int(assignments[building_key]) for building_key in building_keys}

    def _get_agent_weights(self, agent) -> Dict[str, List[np.ndarray]]:
        if not hasattr(agent, "_fed_networks"):
            raise ValueError("Agent does not contain '_fed_networks'.")

        return {
            key: [np.array(w, copy=True) for w in network.get_weights()]
            for key, network in agent._fed_networks.items()
        }

    def _set_agent_weights(self, agent, weights: Dict[str, List[np.ndarray]]) -> None:
        if not hasattr(agent, "_fed_networks"):
            raise ValueError("Agent does not contain '_fed_networks'.")

        for key, network in agent._fed_networks.items():
            if key not in weights:
                raise ValueError(f"Missing weights for key '{key}'.")
            network.set_weights(weights[key])

    def _clone_weights(self, weights: Dict[str, List[np.ndarray]]) -> Dict[str, List[np.ndarray]]:
        return {
            key: [np.array(tensor, copy=True) for tensor in value]
            for key, value in weights.items()
        }

    def _run_local_steps(self, bundle: Dict[str, Any], num_steps: int) -> None:
        if self.algorithm in {"DDPG", "TD3", "SAC"}:
            bundle["time_step"] = bundle["train_env"].reset()
            bundle["policy_state"] = bundle["agent"].collect_policy.get_initial_state(
                bundle["train_env"].batch_size
            )

            for _ in range(int(num_steps)):
                bundle["time_step"], bundle["policy_state"] = bundle["collect_driver"].run(
                    time_step=bundle["time_step"],
                    policy_state=bundle["policy_state"],
                )
                experience, _ = next(bundle["iterator"])
                bundle["agent"].train(experience)

        elif self.algorithm == "PPO":
            for _ in range(int(num_steps)):
                bundle["replay_buffer"].clear()
                bundle["collect_driver"].run(
                    time_step=bundle["train_env"].reset(),
                    policy_state=bundle["agent"].collect_policy.get_initial_state(
                        bundle["train_env"].batch_size
                    ),
                )
                experience = bundle["replay_buffer"].gather_all()
                bundle["agent"].train(experience)

        else:
            raise ValueError(f"Unsupported algorithm: {self.algorithm}")

    def _aggregate_models(
        self,
        global_weights: Dict[str, List[np.ndarray]],
        local_weights_list: List[Dict[str, List[np.ndarray]]],
        round_idx: int,
        cluster_id: int,
    ) -> Dict[str, List[np.ndarray]]:
        """
        Single aggregation function.

        Steps
        -----
        1. compute client updates relative to the cluster global model
        2. optionally clip each client update
        3. average the updates
        4. optionally add DP noise to the averaged update
        5. apply the averaged update to the cluster global model
        """
        if len(local_weights_list) == 0:
            raise ValueError("Cannot aggregate an empty list of local models.")

        aggregated_update: Dict[str, List[np.ndarray]] = {}
        for key in global_weights.keys():
            aggregated_update[key] = [np.zeros_like(tensor) for tensor in global_weights[key]]

        for local_weights in local_weights_list:
            client_update: Dict[str, List[np.ndarray]] = {}

            for key in global_weights.keys():
                client_update[key] = [
                    local_tensor - global_tensor
                    for local_tensor, global_tensor in zip(local_weights[key], global_weights[key])
                ]

            if self.clipping_enabled:
                total_sq_norm = 0.0
                for key in client_update:
                    for tensor in client_update[key]:
                        total_sq_norm += float(np.sum(np.square(tensor)))
                total_norm = float(np.sqrt(total_sq_norm))

                if total_norm > self.clip_norm:
                    scale = self.clip_norm / max(total_norm, 1e-12)
                    for key in client_update:
                        client_update[key] = [scale * tensor for tensor in client_update[key]]

            for key in aggregated_update:
                for tensor_idx in range(len(aggregated_update[key])):
                    aggregated_update[key][tensor_idx] += client_update[key][tensor_idx]

        num_clients = len(local_weights_list)
        for key in aggregated_update:
            aggregated_update[key] = [tensor / num_clients for tensor in aggregated_update[key]]

        if self.differential_privacy and self.dp_noise_multiplier > 0.0:
            rng = np.random.default_rng(self.base_seed + 10000 * round_idx + 100 * int(cluster_id))
            noise_std = self.dp_noise_multiplier * self.clip_norm / max(num_clients, 1)

            for key in aggregated_update:
                for tensor_idx in range(len(aggregated_update[key])):
                    noise = rng.normal(
                        loc=0.0,
                        scale=noise_std,
                        size=aggregated_update[key][tensor_idx].shape,
                    ).astype(aggregated_update[key][tensor_idx].dtype)
                    aggregated_update[key][tensor_idx] = aggregated_update[key][tensor_idx] + noise

        new_global_weights = self._clone_weights(global_weights)
        for key in new_global_weights:
            for tensor_idx in range(len(new_global_weights[key])):
                new_global_weights[key][tensor_idx] = (
                    new_global_weights[key][tensor_idx] + aggregated_update[key][tensor_idx]
                )

        return new_global_weights

    def run(self, prepared: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        self._set_all_seeds(self.base_seed)

        building_keys = list(prepared["dataset"]["train"].keys())
        cluster_assignments = self._load_cluster_assignments(building_keys)

        bundles: Dict[str, Dict[str, Any]] = {}

        for building_idx, building_key in enumerate(building_keys, start=1):
            seed = self.base_seed + 1000 * building_idx

            train_forecast = (
                prepared["forecast_tables"]["train"][building_key]
                if prepared["forecast_tables"] is not None
                else None
            )
            test_forecast = (
                prepared["forecast_tables"]["test"][building_key]
                if prepared["forecast_tables"] is not None
                else None
            )

            train_env = self._build_tf_env(
                data=prepared["dataset"]["train"][building_key],
                bess_meta=prepared["bess_meta"]["train"][building_key],
                obs_stats=prepared["obs_stats"][building_key],
                forecast_df=train_forecast,
                track_history=False,
                seed=seed,
            )
            test_env = self._build_tf_env(
                data=prepared["dataset"]["test"][building_key],
                bess_meta=prepared["bess_meta"]["train"][building_key],
                obs_stats=prepared["obs_stats"][building_key],
                forecast_df=test_forecast,
                track_history=True,
                seed=seed,
            )

            agent = self._create_agent(train_env)

            if self.algorithm in {"DDPG", "TD3", "SAC"}:
                collect_driver, iterator, time_step, policy_state = self._create_off_policy_pipeline(agent, train_env)
                bundles[building_key] = {
                    "seed": seed,
                    "cluster": int(cluster_assignments[building_key]),
                    "train_env": train_env,
                    "test_env": test_env,
                    "agent": agent,
                    "collect_driver": collect_driver,
                    "iterator": iterator,
                    "time_step": time_step,
                    "policy_state": policy_state,
                }

            elif self.algorithm == "PPO":
                replay_buffer, collect_driver = self._create_ppo_pipeline(agent, train_env)
                bundles[building_key] = {
                    "seed": seed,
                    "cluster": int(cluster_assignments[building_key]),
                    "train_env": train_env,
                    "test_env": test_env,
                    "agent": agent,
                    "collect_driver": collect_driver,
                    "replay_buffer": replay_buffer,
                }

            else:
                raise ValueError(f"Unsupported algorithm: {self.algorithm}")

        first_building = building_keys[0]
        initial_global_weights = self._get_agent_weights(bundles[first_building]["agent"])

        unique_clusters = sorted(set(cluster_assignments.values()))
        cluster_global_weights = {
            cluster_id: self._clone_weights(initial_global_weights)
            for cluster_id in unique_clusters
        }

        training_start = time.perf_counter()

        for round_idx in range(self.fed_rounds):
            for cluster_id in unique_clusters:
                cluster_members = [
                    building_key
                    for building_key in building_keys
                    if cluster_assignments[building_key] == cluster_id
                ]

                if len(cluster_members) == 0:
                    continue

                local_weights_list = []

                for building_key in cluster_members:
                    bundle = bundles[building_key]

                    self._set_agent_weights(bundle["agent"], cluster_global_weights[cluster_id])

                    local_train_units = (
                        self.local_steps_per_round
                        if self.algorithm in {"DDPG", "TD3", "SAC"}
                        else self.ppo_local_updates_per_round
                    )
                    self._run_local_steps(bundle, local_train_units)

                    local_weights_list.append(self._get_agent_weights(bundle["agent"]))

                cluster_global_weights[cluster_id] = self._aggregate_models(
                    global_weights=cluster_global_weights[cluster_id],
                    local_weights_list=local_weights_list,
                    round_idx=round_idx,
                    cluster_id=cluster_id,
                )

        training_seconds = time.perf_counter() - training_start

        rows = []
        histories: Dict[str, pd.DataFrame] = {}

        for building_key in building_keys:
            bundle = bundles[building_key]
            cluster_id = bundle["cluster"]

            self._set_agent_weights(bundle["agent"], cluster_global_weights[cluster_id])

            if self.local_retraining_steps > 0:
                retraining_units = (
                    self.local_retraining_steps
                    if self.algorithm in {"DDPG", "TD3", "SAC"}
                    else self.ppo_local_retraining_updates
                )
                self._run_local_steps(bundle, retraining_units)

            eval_result = self._evaluate_policy(bundle["test_env"], bundle["agent"].policy)
            py_env = eval_result["PyEnv"]

            runtime_seconds = training_seconds + float(eval_result["EvaluationRuntimeSeconds"])

            row = py_env.summary(
                agent_name=f"FL_{self.algorithm}",
                building_key=building_key,
                runtime_seconds=runtime_seconds,
                training_seconds=training_seconds,
                extra_metrics={
                    "Cluster": int(cluster_id),
                    "Seed": int(bundle["seed"]),
                    "FedRounds": int(self.fed_rounds),
                    "LocalStepsPerRound": int(self.local_steps_per_round),
                    "ClusteredAggregation": bool(self.clustered_aggregation),
                    "ClippingEnabled": bool(self.clipping_enabled),
                    "ClipNorm": float(self.clip_norm),
                    "DifferentialPrivacy": bool(self.differential_privacy),
                    "DpNoiseMultiplier": float(self.dp_noise_multiplier),
                    "LocalRetrainingSteps": int(self.local_retraining_steps),
                },
            )

            rows.append(row)
            histories[building_key] = eval_result["History"]

        result_df = pd.DataFrame(rows).copy()
        result_df["building_id"] = result_df["Building"].str.extract(r"(\d+)").astype(int)
        result_df = (
            result_df.sort_values(["Cluster", "building_id"])
            .drop(columns="building_id")
            .reset_index(drop=True)
        )

        return result_df, histories

    
class RunExperiments:
    """
    Thin coordinator that now owns

    1. data loading
    2. metadata preparation
    3. forecast table generation
    4. execution order
    5. result saving
    6. final comparison table

    This is the only coordinator class so the structure stays compact.
    """

    def __init__(
        self,
        num_buildings: int = 20,
        ecoPriority: float = 0.0,
        path_energy_data: str = "data/Final_Energy_dataset.csv.xz",
        results_dir: str = "results/benchmark_suite",
        feed_in_price: float = 0.076,
        power_grid_kw: float = 25.0,
        init_charge_kwh: float = 0.0,
        penalty_factor: float = 1.0,
        solver_name: Optional[str] = None,
        forecast_horizon: int = 48,
        forecast_base_seed: int = 42,
        forecast_prosumption_config: Optional[Dict[str, Any]] = None,
        forecast_price_config: Optional[Dict[str, Any]] = None,
        forecast_emission_config: Optional[Dict[str, Any]] = None,
        ll_batch_size: int = 128,
        ll_replay_buffer_capacity: int = 200000,
        ll_initial_collect_steps: int = 10000,
        ll_collect_steps_per_iteration: int = 1,
        ll_num_iterations: int = 20000,
        fl_fed_rounds: int = 10,
        fl_local_steps_per_round: int = 5000,
        fl_clustered_aggregation: bool = True,
        fl_cluster_file_path: Optional[str] = None,
        fl_clipping_enabled: bool = False,
        fl_clip_norm: float = 1.0,
        fl_differential_privacy: bool = False,
        fl_dp_noise_multiplier: float = 0.0,
        fl_local_retraining_steps: int = 0,
        ll_algorithms: Optional[List[str]] = None,
        fl_algorithms: Optional[List[str]] = None,
        ppo_collect_steps: int = 2048,
        ppo_num_epochs: int = 10,
        actor_learning_rate: float = 1e-4,
        critic_learning_rate: float = 3e-4,
        price_feature_mode: str = "local_absolute",
        local_price_lower_q: float = 0.05,
        local_price_upper_q: float = 0.95,
        run_models: Optional[List[str]] = None,
    ):
        self.num_buildings = int(num_buildings)
        self.ecoPriority = float(ecoPriority)
        self.path_energy_data = path_energy_data
        self.results_dir = results_dir

        self.feed_in_price = float(feed_in_price)
        self.power_grid_kw = float(power_grid_kw)
        self.init_charge_kwh = float(init_charge_kwh)
        self.penalty_factor = float(penalty_factor)
        self.solver_name = solver_name

        self.forecast_horizon = int(forecast_horizon)
        self.forecast_base_seed = int(forecast_base_seed)
        self.forecast_prosumption_config = forecast_prosumption_config
        self.forecast_price_config = forecast_price_config
        self.forecast_emission_config = forecast_emission_config

        self.ll_batch_size = int(ll_batch_size)
        self.ll_replay_buffer_capacity = int(ll_replay_buffer_capacity)
        self.ll_initial_collect_steps = int(ll_initial_collect_steps)
        self.ll_collect_steps_per_iteration = int(ll_collect_steps_per_iteration)
        self.ll_num_iterations = int(ll_num_iterations)

        self.fl_fed_rounds = int(fl_fed_rounds)
        self.fl_local_steps_per_round = int(fl_local_steps_per_round)

        self.fl_clustered_aggregation = bool(fl_clustered_aggregation)
        self.fl_cluster_file_path = fl_cluster_file_path
        self.fl_clipping_enabled = bool(fl_clipping_enabled)
        self.fl_clip_norm = float(fl_clip_norm)
        self.fl_differential_privacy = bool(fl_differential_privacy)
        self.fl_dp_noise_multiplier = float(fl_dp_noise_multiplier)
        self.fl_local_retraining_steps = int(fl_local_retraining_steps)

        self.ll_algorithms = [alg.upper() for alg in (ll_algorithms or ["DDPG"])]
        self.fl_algorithms = [alg.upper() for alg in (fl_algorithms or ["DDPG"])]
        self.ppo_collect_steps = int(ppo_collect_steps)
        self.ppo_num_epochs = int(ppo_num_epochs)

        self.actor_learning_rate = float(actor_learning_rate)
        self.critic_learning_rate = float(critic_learning_rate)
        self.price_feature_mode = str(price_feature_mode)
        self.local_price_lower_q = float(local_price_lower_q)
        self.local_price_upper_q = float(local_price_upper_q)

        self.run_models = run_models or ["noBess", "RuleBess", "MIP", "MPC", "LL_DDPG", "FL_DDPG"]

        self.prepared: Optional[Dict[str, Any]] = None

    def _split_series(self, series: pd.Series) -> Dict[str, pd.Series]:
        return {
            "train": series.iloc[:EnergyManagementEnv.TRAIN_END].reset_index(drop=True),
            "eval": series.iloc[EnergyManagementEnv.TRAIN_END:EnergyManagementEnv.EVAL_END].reset_index(drop=True),
            "test": series.iloc[EnergyManagementEnv.EVAL_END:EnergyManagementEnv.TEST_END].reset_index(drop=True),
        }

    def _forecast_cfg(self, cfg: Optional[Dict[str, Any]]) -> Dict[str, Any]:
        out = {
            "mode": "perfect",  # perfect | noise | file
            "noise_strength": 0.0,
            "rho": 0.95,
            "horizon_growth": 1.0,
            "bias_strength": 0.0,
            "file_path": None,
            "model_name": None,
            "pred_col_template": "y_pred_bld{building_idx}_{model_name}",
            "fill_missing_with_actual": True,
        }
        if cfg is not None:
            out.update(cfg)
        return out

    def _load_prediction_series(
        self,
        file_path: str,
        building_idx: int,
        model_name: str,
        pred_col_template: str,
        full_index: pd.Index,
    ) -> pd.Series:
        pred_df = pd.read_csv(file_path)
        if "timestamp" not in pred_df.columns:
            raise ValueError("Forecast file must contain a 'timestamp' column.")

        pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"])
        pred_df = pred_df.set_index("timestamp").sort_index()

        pred_col = pred_col_template.format(building_idx=building_idx, model_name=model_name)
        if pred_col not in pred_df.columns:
            raise ValueError(f"Column '{pred_col}' not found in '{file_path}'.")

        pred_series = pred_df[pred_col].astype(float)
        return pred_series.reindex(full_index)

    def _draw_ar1_noise(self, horizon: int, rho: float, rng: np.random.Generator) -> np.ndarray:
        rho = float(np.clip(rho, -0.999, 0.999))
        z = np.zeros(horizon, dtype=np.float64)
        z[0] = rng.normal()
        innovation_std = np.sqrt(max(1.0 - rho * rho, 1e-8))
        for h in range(1, horizon):
            z[h] = rho * z[h - 1] + innovation_std * rng.normal()
        return z

    def _apply_noise(
        self,
        base_window: np.ndarray,
        reference_std: float,
        cfg: Dict[str, Any],
        rng: np.random.Generator,
    ) -> np.ndarray:
        if float(cfg["noise_strength"]) <= 0.0:
            return base_window

        horizon = len(base_window)
        z = self._draw_ar1_noise(horizon, float(cfg["rho"]), rng)
        growth = 1.0 + float(cfg["horizon_growth"]) * np.arange(horizon, dtype=np.float64) / max(horizon - 1, 1)
        noise = z * float(cfg["noise_strength"]) * reference_std * growth
        shared_bias = rng.normal(loc=0.0, scale=float(cfg["bias_strength"]) * reference_std)
        return base_window + noise + shared_bias

    def _build_forecast_matrix(
        self,
        actual_split_series: pd.Series,
        source_split_series: pd.Series,
        cfg: Dict[str, Any],
        rng: np.random.Generator,
    ) -> np.ndarray:
        actual_values = actual_split_series.to_numpy(dtype=np.float64)
        source_values = source_split_series.to_numpy(dtype=np.float64)

        if cfg["fill_missing_with_actual"]:
            source_values = np.where(np.isnan(source_values), actual_values, source_values)

        T = len(actual_values)
        H = self.forecast_horizon
        matrix = np.zeros((T, H), dtype=np.float32)

        reference_std = max(float(np.std(actual_values)), 1e-6)

        for t in range(T):
            start = t + 1
            end = min(T, t + 1 + H)
            future_window = source_values[start:end].copy()

            if len(future_window) == 0:
                fill_value = source_values[t] if t < T else actual_values[-1]
                future_window = np.full(H, fill_value, dtype=np.float64)
            elif len(future_window) < H:
                future_window = np.pad(
                    future_window,
                    (0, H - len(future_window)),
                    mode="constant",
                    constant_values=future_window[-1],
                )

            if cfg["mode"] in {"noise", "file"}:
                future_window = self._apply_noise(future_window, reference_std, cfg, rng)

            matrix[t, :] = future_window.astype(np.float32)

        return matrix

    def _build_forecast_tables(self) -> Optional[Dict[str, Dict[str, pd.DataFrame]]]:
        prosumption_cfg = self._forecast_cfg(self.forecast_prosumption_config)
        price_cfg = self._forecast_cfg(self.forecast_price_config)
        emission_cfg = self._forecast_cfg(self.forecast_emission_config)

        if all(cfg["mode"] == "perfect" and float(cfg["noise_strength"]) == 0.0 for cfg in [prosumption_cfg, price_cfg, emission_cfg]):
            return None

        energy_data = pd.read_csv(self.path_energy_data).fillna(0.0)
        if "Date" not in energy_data.columns:
            raise ValueError("Energy data must contain a 'Date' column for file based forecast alignment.")

        energy_data["Date"] = pd.to_datetime(energy_data["Date"])
        energy_data = energy_data.set_index("Date").sort_index()

        forecast_tables: Dict[str, Dict[str, pd.DataFrame]] = {"train": {}, "eval": {}, "test": {}}

        for building_idx in range(1, self.num_buildings + 1):
            building_key = f"building_{building_idx}"

            actual_prosumption = energy_data[f"load_{building_idx}"] - energy_data[f"pv_{building_idx}"]
            actual_price = energy_data["price"]
            actual_emission = energy_data["emissions"]

            if prosumption_cfg["mode"] == "file":
                source_prosumption = self._load_prediction_series(
                    file_path=prosumption_cfg["file_path"],
                    building_idx=building_idx,
                    model_name=prosumption_cfg["model_name"],
                    pred_col_template=prosumption_cfg["pred_col_template"],
                    full_index=energy_data.index,
                )
            else:
                source_prosumption = actual_prosumption.copy()

            if price_cfg["mode"] == "file":
                source_price = self._load_prediction_series(
                    file_path=price_cfg["file_path"],
                    building_idx=building_idx,
                    model_name=price_cfg["model_name"],
                    pred_col_template=price_cfg["pred_col_template"],
                    full_index=energy_data.index,
                )
            else:
                source_price = actual_price.copy()

            if emission_cfg["mode"] == "file":
                source_emission = self._load_prediction_series(
                    file_path=emission_cfg["file_path"],
                    building_idx=building_idx,
                    model_name=emission_cfg["model_name"],
                    pred_col_template=emission_cfg["pred_col_template"],
                    full_index=energy_data.index,
                )
            else:
                source_emission = actual_emission.copy()

            actual_splits = {
                "prosumption": self._split_series(actual_prosumption),
                "price": self._split_series(actual_price),
                "emission": self._split_series(actual_emission),
            }
            source_splits = {
                "prosumption": self._split_series(source_prosumption),
                "price": self._split_series(source_price),
                "emission": self._split_series(source_emission),
            }

            for split_idx, split_name in enumerate(["train", "eval", "test"]):
                rng = np.random.default_rng(self.forecast_base_seed + 10000 * building_idx + 100 * split_idx)

                pros_matrix = self._build_forecast_matrix(
                    actual_split_series=actual_splits["prosumption"][split_name],
                    source_split_series=source_splits["prosumption"][split_name],
                    cfg=prosumption_cfg,
                    rng=rng,
                )
                price_matrix = self._build_forecast_matrix(
                    actual_split_series=actual_splits["price"][split_name],
                    source_split_series=source_splits["price"][split_name],
                    cfg=price_cfg,
                    rng=rng,
                )
                emission_matrix = self._build_forecast_matrix(
                    actual_split_series=actual_splits["emission"][split_name],
                    source_split_series=source_splits["emission"][split_name],
                    cfg=emission_cfg,
                    rng=rng,
                )

                forecast_tables[split_name][building_key] = pd.DataFrame(
                    {
                        **{f"prosumption_f{i + 1}": pros_matrix[:, i] for i in range(self.forecast_horizon)},
                        **{f"price_f{i + 1}": price_matrix[:, i] for i in range(self.forecast_horizon)},
                        **{f"emission_f{i + 1}": emission_matrix[:, i] for i in range(self.forecast_horizon)},
                    }
                )

        return forecast_tables

    def _prepare(self) -> Dict[str, Any]:
        if self.prepared is not None:
            return self.prepared

        dataset, bess_meta, obs_stats = EnergyManagementEnv.prepare_buildings(
            path_energy_data=self.path_energy_data,
            num_buildings=self.num_buildings,
            ecoPriority=self.ecoPriority,
            feed_in_price=self.feed_in_price,
        )

        forecast_tables = self._build_forecast_tables()

        self.prepared = {
            "dataset": dataset,
            "bess_meta": bess_meta,
            "obs_stats": obs_stats,
            "forecast_tables": forecast_tables,
        }
        return self.prepared

    def _prepare_output_dirs(self, model_name: str) -> str:
        model_dir = os.path.join(self.results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        os.makedirs(os.path.join(model_dir, "histories"), exist_ok=True)
        return model_dir

    def _save_results(
        self,
        model_name: str,
        result_df: pd.DataFrame,
        histories: Dict[str, pd.DataFrame],
    ) -> None:
        model_dir = self._prepare_output_dirs(model_name)

        result_df.to_csv(os.path.join(model_dir, f"{model_name}_results.csv"), index=False)

        for building_key, history_df in histories.items():
            history_df.to_csv(
                os.path.join(model_dir, "histories", f"{building_key}_{model_name}_history.csv"),
                index=False,
            )

        summary = {
            "Model": model_name,
            "NumBuildings": int(len(result_df)),
            "EcoPriority": float(self.ecoPriority),
            "SumTotalProfit": float(result_df["Total Profit"].sum()),
            "SumTotalCost": float(result_df["Total Cost"].sum()),
            "SumTotalEmissions": float(result_df["Total Emissions"].sum()),
            "MeanTotalProfit": float(result_df["Total Profit"].mean()),
            "MeanTotalCost": float(result_df["Total Cost"].mean()),
            "MeanTotalEmissions": float(result_df["Total Emissions"].mean()),
        }

        with open(os.path.join(model_dir, f"{model_name}_summary.json"), "w") as f:
            json.dump(summary, f, indent=2)

    def _build_comparison(self, all_results: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        comparison = pd.DataFrame({"Building": list(self._prepare()["dataset"]["test"].keys())})

        for model_name, result_df in all_results.items():
            keep_cols = result_df[["Building", "Total Profit", "Total Emissions"]].rename(
                columns={
                    "Total Profit": f"{model_name}_Profit",
                    "Total Emissions": f"{model_name}_Emissions",
                }
            )
            comparison = comparison.merge(keep_cols, on="Building", how="left")

        comparison["EcoPriority"] = self.ecoPriority
        comparison["building_id"] = comparison["Building"].str.extract(r"(\d+)").astype(int)
        comparison = comparison.sort_values("building_id").drop(columns=["building_id"]).reset_index(drop=True)
        return comparison

    def run(self) -> Dict[str, Any]:
        prepared = self._prepare()
        os.makedirs(self.results_dir, exist_ok=True)

        all_results: Dict[str, pd.DataFrame] = {}
        all_histories: Dict[str, Dict[str, pd.DataFrame]] = {}

        model_objects = {
            "noBess": NoBess(
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
            ),
            "RuleBess": RuleBess(
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
                forecast_horizon=self.forecast_horizon,
            ),
            "MIP": MIP(
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
                solver_name=self.solver_name,
                enforce_terminal_soe=False,
            ),
            "MPC": MPC(
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
                solver_name=self.solver_name,
                forecast_horizon=self.forecast_horizon,
                terminal_soe_constraint=False,
            ),
        }

        for algorithm in self.ll_algorithms:
            model_objects[f"LL_{algorithm}"] = LL_RL(
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
                base_seed=self.forecast_base_seed,
                batch_size=self.ll_batch_size,
                replay_buffer_capacity=self.ll_replay_buffer_capacity,
                initial_collect_steps=self.ll_initial_collect_steps,
                collect_steps_per_iteration=self.ll_collect_steps_per_iteration,
                num_iterations=self.ll_num_iterations,
                actor_learning_rate=self.actor_learning_rate,
                critic_learning_rate=self.critic_learning_rate,
                price_feature_mode=self.price_feature_mode,
                local_price_lower_q=self.local_price_lower_q,
                local_price_upper_q=self.local_price_upper_q,
                algorithm=algorithm,
                ppo_collect_steps=self.ppo_collect_steps,
                ppo_num_epochs=self.ppo_num_epochs,
            )

        for algorithm in self.fl_algorithms:
            model_objects[f"FL_{algorithm}"] = FL_RL(
                ecoPriority=self.ecoPriority,
                feed_in_price=self.feed_in_price,
                power_grid_kw=self.power_grid_kw,
                init_charge_kwh=self.init_charge_kwh,
                penalty_factor=self.penalty_factor,
                base_seed=self.forecast_base_seed,
                batch_size=self.ll_batch_size,
                replay_buffer_capacity=self.ll_replay_buffer_capacity,
                initial_collect_steps=self.ll_initial_collect_steps,
                collect_steps_per_iteration=self.ll_collect_steps_per_iteration,
                num_iterations=self.ll_num_iterations,
                actor_learning_rate=self.actor_learning_rate,
                critic_learning_rate=self.critic_learning_rate,
                price_feature_mode=self.price_feature_mode,
                local_price_lower_q=self.local_price_lower_q,
                local_price_upper_q=self.local_price_upper_q,
                fed_rounds=self.fl_fed_rounds,
                local_steps_per_round=self.fl_local_steps_per_round,
                clustered_aggregation=self.fl_clustered_aggregation,
                cluster_file_path=self.fl_cluster_file_path,
                clipping_enabled=self.fl_clipping_enabled,
                clip_norm=self.fl_clip_norm,
                differential_privacy=self.fl_differential_privacy,
                dp_noise_multiplier=self.fl_dp_noise_multiplier,
                local_retraining_steps=self.fl_local_retraining_steps,
                algorithm=algorithm,
                ppo_collect_steps=self.ppo_collect_steps,
                ppo_num_epochs=self.ppo_num_epochs,
            )

        for model_name in self.run_models:
            print(model_name)
            result_df, histories = model_objects[model_name].run(prepared)
            all_results[model_name] = result_df
            all_histories[model_name] = histories
            self._save_results(model_name, result_df, histories)

        comparison_df = self._build_comparison(all_results)
        comparison_df.to_csv(os.path.join(self.results_dir, "comparison.csv"), index=False)

        return {
            "prepared": prepared,
            "results": all_results,
            "histories": all_histories,
            "comparison": comparison_df,
        }



In [ ]:
# FLavg
runner = RunExperiments(
    num_buildings=5, ecoPriority=0.5, results_dir="results/RaspberryComputationalTime_EcoPriority05",
    feed_in_price=0.05,
    forecast_prosumption_config={
        "mode": "file", "file_path": "utils/Forecasts/LL_Prosumption_Ausgrid_forecasts.csv.xz",
        "model_name": "softdensemoe", "pred_col_template": "y_pred_bld{building_idx}_{model_name}",
        "noise_strength": 0.0,},
    forecast_price_config={"mode": "perfect"},
    forecast_emission_config={
        "mode": "file", "file_path": "utils/Forecasts/LL_Emissions_forecasts.csv.xz",
        "model_name": "softdensemoe", "pred_col_template": "y_pred_bld1_softdensemoe",
        "noise_strength": 0.0,},

    ll_batch_size=128, ll_replay_buffer_capacity=200000, ll_initial_collect_steps=100000,
    ll_collect_steps_per_iteration=1, ll_num_iterations=10000,

    fl_fed_rounds=5, fl_local_steps_per_round=10000, 
    fl_clustered_aggregation=True, fl_cluster_file_path="utils/Clusters/Clusters_ausgrid_20first_k7.csv",
    fl_clipping_enabled=False, fl_clip_norm=1.0,
    fl_differential_privacy=False, fl_dp_noise_multiplier=0.0,
    fl_local_retraining_steps=10000,

    actor_learning_rate=1e-4, critic_learning_rate=3e-4, price_feature_mode="local_absolute",  # local_absolute | local_delta
    local_price_lower_q=0.05, local_price_upper_q=0.95,
    
    ll_algorithms=["DDPG", "TD3", "SAC",], #"PPO"],
    fl_algorithms=["DDPG", "TD3", "SAC",], # "PPO"],
    run_models=[
        "noBess", "RuleBess", "MIP", "MPC", 
        "LL_DDPG", "LL_TD3", "LL_SAC",
        #"FL_DDPG", "FL_TD3",
        #"FL_SAC",
    ],
)

results = runner.run()

noBess


KeyboardInterrupt: 